# Light Classifier (TFLite Micro) — extra da Aula 4

Modelo regressor que aprende a recuperar a luminosidade real (`lux_norm` em [0,1]) a partir da leitura de um LDR em divisor de tensão lida pelo ADC do ESP32-S3 (`adc_norm` em [0,1]).

Diferenças em relação ao Hello World:
- **Sensor real** (LDR + resistor) em vez de `x` artificial.
- **Dataset sintético novo** com não-linearidade tipo `tanh` para simular saturação do sensor.
- **Aplicação diferente**: classificação em 4 zonas de luminosidade (Escuro / Penumbra / Claro / Muito Claro) feita no firmware a partir do score do regressor.

Mantém o mesmo backbone (`Dense(16)→Dense(16)→Dense(1)`, op `FullyConnected`) — então o `MicroMutableOpResolver<1>` do firmware não muda.

In [ ]:
!pip -q install tensorflow

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

## 1) Dataset sintético

O ADC do ESP32-S3 cobre **`adc_norm ∈ [0, 1]` por inteiro** (0..4095 raw). O dataset precisa cobrir a faixa completa, senão o modelo extrapola mal nos extremos.

Mapping ground-truth: `lux = sqrt(adc_norm)` — curva côncava (cresce rápido em pouca luz, satura em muita) que simula a resposta logarítmica de um LDR real.

| `adc_norm` | `lux` esperado | classe        |
| ---------- | -------------- | ------------- |
| 0.00       | 0.00           | ESCURO        |
| 0.06       | 0.25           | limiar ESCURO/PENUMBRA |
| 0.25       | 0.50           | limiar PENUMBRA/CLARO  |
| 0.56       | 0.75           | limiar CLARO/MUITO_CLARO |
| 1.00       | 1.00           | MUITO_CLARO   |

In [ ]:
N = 2000
# adc_norm cobre [0,1] uniformemente (faixa real do ADC do ESP32-S3).
adc_norm = np.random.uniform(0.0, 1.0, N).astype(np.float32)
# lux real segue uma curva sqrt (concava). Adiciona ruido pequeno.
lux_true = (np.sqrt(adc_norm) + np.random.normal(0, 0.02, N)).clip(0, 1).astype(np.float32)

plt.figure(figsize=(6,4))
plt.scatter(adc_norm, lux_true, s=4, alpha=0.6, label='dataset')
xs = np.linspace(0, 1, 100)
plt.plot(xs, np.sqrt(xs), 'r--', label=r'$\sqrt{adc}$')
plt.xlabel('ADC normalizado (input)')
plt.ylabel('lux real (target)')
plt.title('Dataset sintetico — cobre [0,1] por inteiro')
plt.legend(); plt.grid(True); plt.show()

## 2) Treinar o modelo

Mesma arquitetura do Hello World original (Dense(16)→Dense(16)→Dense(1)) — só muda o dataset.

In [ ]:
def create_model():
    m = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation='relu', input_shape=(1,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1),
    ])
    m.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return m

model = create_model()
history = model.fit(
    adc_norm, lux_true,
    epochs=300,
    batch_size=64,
    validation_split=0.2,
    verbose=2,
)

## 3) Exportar SavedModel + converter para TFLite int8

Mesma quantização pós-treinamento int8 do exemplo original (input/output `int8`).

In [ ]:
SAVE_DIR  = '/content/light_classifier_models'
SAVED_DIR = SAVE_DIR + '/saved_model'
os.makedirs(SAVED_DIR, exist_ok=True)

if hasattr(model, 'export'):
    model.export(SAVED_DIR)
else:
    tf.saved_model.save(model, SAVED_DIR)

def representative_dataset():
    for i in range(500):
        yield [adc_norm[i].reshape(1, 1).astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_DIR)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8
converter.representative_dataset = representative_dataset
tflite_int8 = converter.convert()

out_path = SAVE_DIR + '/light_classifier_int8.tflite'
with open(out_path, 'wb') as f:
    f.write(tflite_int8)
print(f'OK: {out_path} ({len(tflite_int8)} bytes)')

## 4) Avaliar o modelo int8 e plotar

In [ ]:
import tensorflow.lite as tflite_interp
interp = tflite_interp.Interpreter(model_content=tflite_int8)
interp.allocate_tensors()
in_d  = interp.get_input_details()[0]
out_d = interp.get_output_details()[0]
in_scale,  in_zp  = in_d['quantization']
out_scale, out_zp = out_d['quantization']

preds = np.empty(N, dtype=np.float32)
for i, x in enumerate(adc_norm):
    x_q = np.round(x / in_scale + in_zp).clip(-128, 127).astype(np.int8).reshape(1, 1)
    interp.set_tensor(in_d['index'], x_q)
    interp.invoke()
    y_q = interp.get_tensor(out_d['index'])[0, 0]
    preds[i] = (float(y_q) - out_zp) * out_scale

mae  = float(np.mean(np.abs(preds - lux_true)))
rmse = float(np.sqrt(np.mean((preds - lux_true) ** 2)))
print(f'Quantized INT8 — MAE={mae:.4f} RMSE={rmse:.4f}')

order = np.argsort(adc_norm)
plt.figure(figsize=(6,4))
plt.scatter(adc_norm, lux_true, s=4, alpha=0.4, label='ground truth')
plt.plot(adc_norm[order], preds[order], 'r-', linewidth=2, label='int8 predictions')
plt.xlabel('ADC normalizado'); plt.ylabel('lux predito')
plt.title(f'Light classifier INT8 — MAE={mae:.4f}')
plt.legend(); plt.grid(True); plt.show()

# Distribuicao por classes (mesmos limiares do firmware)
for lo, hi, name in [(0.00, 0.25, 'ESCURO'), (0.25, 0.50, 'PENUMBRA'),
                     (0.50, 0.75, 'CLARO'),  (0.75, 1.01, 'MUITO_CLARO')]:
    n_true = int(((lux_true >= lo) & (lux_true < hi)).sum())
    n_pred = int(((preds    >= lo) & (preds    < hi)).sum())
    print(f'{name:<12} ground_truth={n_true:4d} | int8_pred={n_pred:4d}')

## 5) Baixar o `.tflite`

Salva em `Downloads/`. Depois rode localmente:
```powershell
cd "C:\Users\rosej\OneDrive\Desktop\IA Embarcada\Aula 4\light_classifier_tflite"
python tflite_to_cc.py "$env:USERPROFILE\Downloads\light_classifier_int8.tflite"
```
Isso gera `main/model.cc` e o projeto fica pronto para `idf.py build`.

In [ ]:
from google.colab import files
files.download(out_path)